# Frontend HTML para etiquetado humano

**Objetivo:** crear una interfaz local para etiquetar chunks de transcripciones con categorias de moderacion de contenido.

El frontend se ejecuta como archivo HTML. No usa servidor, API ni base de datos externa. Puede cargar los chunks embebidos o desde un archivo local, pero exporta un JSONL ligero: conserva `chunk_id`, etiquetas y metadatos de anotación, sin duplicar el texto.

In [1]:
from pathlib import Path
import itertools
import json
import re
import pandas as pd
from collections import Counter

ROOT         = Path('..').resolve()
FRONTEND_DIR = ROOT / 'Cuadernos' / 'frontend'
PROCESSED_DIR = ROOT / 'datos' / 'processed'
LABELED_DIR  = ROOT / 'datos' / 'etiquetado'

for d in [FRONTEND_DIR, LABELED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Frontend   :', FRONTEND_DIR)
print('Processed  :', PROCESSED_DIR)
print('Etiquetado :', LABELED_DIR)


Frontend   : D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\Cuadernos\frontend
Processed  : D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\processed
Etiquetado : D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado


## 1. Taxonomia de etiquetado (fundamentada en expertos peruanos)

La taxonomia es **multi-etiqueta** y no mutuamente excluyente, salvo las dos etiquetas seguras entre sí y `seguro` vs. etiquetas de daño. Toda anotación exportada lleva al menos una etiqueta principal; los flags solo acompañan daño y nunca lo reemplazan.

**Base academica:**
- Portocarrero (2009); Vich (2018): racismo negado — opera disfrazado de criterio de educacion o cultura.
- Zavala & Zariquiey (2007); Zavala & Back (2017): racismo encubierto y racismo linguistico.
- Almeida & Zavala (2022): motoseo y ortografia andina como discriminacion en redes sociales.
- Callirgos (1993); Branez Medina (2012): clasismo y racismo como categorias inseparables en Peru.
- Monge-Olivarria & Guerra-Corrales (2023): feminizacion como insulto en Twitter peruano.
- Thakur / CDT (2025): falla de sistemas automaticos con espanol andino y quechua mezclado.

**Proceso de decision:** 7 pasos con preguntas clave por subcategoria y 13 ejemplos anotados disponibles en:
`modelos/skills/clasificacion_moderacion_peru.md`

In [2]:
# Taxonomia peruana contextualizada — multi-etiqueta, no mutuamente excluyente
LABEL_SECTIONS = {
    'SEGURO': [
        'seguro',                    # informativo, descriptivo o humor sin ataque
        'seguro_ironia_marcada',     # parodia cuyo blanco NO es un grupo humano
    ],
    'RACISMO_DISCRIMINACION': [
        'racismo_etnico_explicito',  # serrano, cholo, negro, indio derogatorio (Callirgos, 1993)
        'racismo_linguistico',       # burla del acento andino, motoseo (Almeida & Zavala, 2022)
        'clasismo_racial',           # amixer, huachafa, chusma (Branez Medina, 2012)
        'discriminacion_regional',   # Lima vs. provincias, centralismo discriminatorio
        'racismo_encubierto',        # criterio de "cultura/educacion" para segregar (Zavala & Zariquiey, 2007)
    ],
    'ACOSO': [
        'misoginia_acoso_genero',    # insultos por ser mujer, feminizacion como insulto (Monge-Olivarria, 2023)
        'homofobia_transfobia',      # insultos/amenazas contra LGBTQ+
        'acoso_personal',            # ataque a persona identificable, doxeo
        'amenaza_directa',           # expresion explicita de intencion de dano
    ],
    'CONTENIDO_SEXUAL': [
        'sexual_explicito',          # descripcion grafica sin proposito informativo
        'sexual_cosificacion',       # sexualizacion de personas como objetos
        'sexual_no_consensual',      # revenge porn, grabacion sin consentimiento
    ],
}

# Flags transversales: se suman a categorias de dano, nunca las reemplazan
FLAGS = [
    'ironia_ambigua',       # no se distingue ironia critica de dano genuino (Vich, 2018)
    'humor_encubridor',     # el hablante usa humor para negar el dano (Branez Medina, 2012)
    'contexto_necesario',   # chunk aislado insuficiente; requiere ver el video (Thakur / CDT, 2025)
]

# Lista plana de etiquetas (sin flags) — usada por el frontend HTML
labels = [etiqueta for seccion in LABEL_SECTIONS.values() for etiqueta in seccion]
all_labels_and_flags = labels + FLAGS
SAFE_LABELS = frozenset(LABEL_SECTIONS['SEGURO'])
DAMAGE_LABELS = frozenset(labels) - SAFE_LABELS
ALLOWED_LABELS = frozenset(labels)
ALLOWED_FLAGS = frozenset(FLAGS)

print(f'Categorias principales : {len(LABEL_SECTIONS)}')
print(f'Etiquetas de dano/seguro: {len(labels)}')
print(f'Flags transversales    : {len(FLAGS)}')
print()
for cat, etiquetas in LABEL_SECTIONS.items():
    print(f'{cat}:')
    for e in etiquetas:
        print(f'  - {e}')
print('\nFLAGS:')
for f in FLAGS:
    print(f'  - {f}')


Categorias principales : 4
Etiquetas de dano/seguro: 14
Flags transversales    : 3

SEGURO:
  - seguro
  - seguro_ironia_marcada
RACISMO_DISCRIMINACION:
  - racismo_etnico_explicito
  - racismo_linguistico
  - clasismo_racial
  - discriminacion_regional
  - racismo_encubierto
ACOSO:
  - misoginia_acoso_genero
  - homofobia_transfobia
  - acoso_personal
  - amenaza_directa
CONTENIDO_SEXUAL:
  - sexual_explicito
  - sexual_cosificacion
  - sexual_no_consensual

FLAGS:
  - ironia_ambigua
  - humor_encubridor
  - contexto_necesario


## 2. Frontend HTML externo

El archivo del frontend se mantiene fuera del notebook. Flujo recomendado:

1. Elegir una variante: `etiquetado_humano.html` (dataset embebido) o `etiquetado_humano_sin_datos.html` (sin registros).
2. En la variante sin datos, pulsar **Cargar JSONL/JSON** y seleccionar localmente `datos/processed/chunks_para_etiquetar.jsonl`; el navegador no lo envía a ningún servidor.
3. **Configurar anotador** en la parte superior: tipo (Humano / LLM), iniciales (máx. 3 letras) y modelo si aplica.
4. Etiquetar fragmentos presentados en **orden aleatorio** con la taxonomía peruana (multi-etiqueta + flags). La interfaz muestra título/canal y permite desplegar los chunks vecinos del mismo video, como la segunda pasada LLM. También impide combinar seguro con daño, combinar las dos etiquetas seguras o usar flags sin una categoría de daño.
5. Exportar — el archivo se nombra automáticamente `<iniciales>_labeled_chunks.jsonl` y contiene solo `chunk_id`, etiquetas y metadatos; no repite el texto ni los demás campos del chunk.
6. Depositar en `datos/etiquetado/` para consolidación multi-anotador (sección 4).

Un mismo chunk puede ser anotado por humanos y LLMs en sesiones separadas.
La clave única por registro es `(chunk_id, annotator_id)`.
Las iniciales y metadatos del anotador persisten entre chunks y recargas del navegador.


In [3]:
from pathlib import Path

html_path = FRONTEND_DIR / 'etiquetado_humano.html'
if not html_path.exists():
    raise FileNotFoundError(f'No existe el frontend: {html_path}')

print('Frontend disponible en:', html_path)


Frontend disponible en: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\Cuadernos\frontend\etiquetado_humano.html


## 3. Archivos de contexto para clasificación con LLM

Para que un LLM comercial clasifique chunks automáticamente, carga estos tres archivos:

| Rol en el prompt | Archivo |
|---|---|
| **System message** (taxonomía + 7 pasos + 13 ejemplos) | `modelos/skills/clasificacion_moderacion_peru.md` |
| Referencia de etiquetas y fuentes | `datos/processed/taxonomia_moderacion.csv` |
| Chunks a clasificar | `datos/processed/chunks_para_etiquetar.jsonl` |

**Instrucción de uso:**
- Cargar el skill `.md` completo como `system message`.
- Enviar cada chunk como `user message`: `{"chunk_id": "...", "text": "..."}`.
- Solicitar respuesta en el JSON ligero definido en la sección 4: no copiar `text` ni metadatos del chunk.
- Guardar cada respuesta en `datos/etiquetado/<modelo>_labeled_chunks.jsonl` con `annotator_type="llm"`.

In [4]:
CONTEXT_FILES = {
    'system_prompt (skill)': ROOT / 'modelos' / 'skills' / 'clasificacion_moderacion_peru.md',
    'taxonomia_csv':         PROCESSED_DIR / 'taxonomia_moderacion.csv',
    'chunks_jsonl':          PROCESSED_DIR / 'chunks_para_etiquetar.jsonl',
    'frontend_html':         FRONTEND_DIR  / 'etiquetado_humano.html',
    'frontend_sin_datos':    FRONTEND_DIR  / 'etiquetado_humano_sin_datos.html',
}

print('Archivos necesarios:')
for rol, path in CONTEXT_FILES.items():
    estado = '✓ existe' if path.exists() else '✗ no encontrado'
    print(f'  [{estado}]  {rol}')
    print(f'             {path}')


Archivos necesarios:
  [✓ existe]  system_prompt (skill)
             D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\modelos\skills\clasificacion_moderacion_peru.md
  [✓ existe]  taxonomia_csv
             D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\processed\taxonomia_moderacion.csv
  [✓ existe]  chunks_jsonl
             D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\processed\chunks_para_etiquetar.jsonl
  [✓ existe]  frontend_html
             D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\Cuadernos\frontend\etiquetado_humano.html
  [✓ existe]  frontend_sin_datos
             D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\Cuadernos\frontend\etiquetado_humano_sin_datos.html


In [ ]:
# ── Esquema de salida del etiquetado ─────────────────────────────────────────
# Clave única por registro: (chunk_id, annotator_id)
# El mismo chunk puede tener N filas, una por anotador (humano o LLM).

ANNOTATED_FIELDS = {
    # Referencia al chunk original. El texto se recupera por chunk_id.
    'chunk_id':        'str   — identificador único del chunk',
    # Etiquetas
    'labels':          'list  — no vacía; una segura o una/más categorías de daño',
    'flags':           'list  — solo acompañan categorías de daño',
    'needs_review':    'bool  — True si hay flags o incertidumbre explícita',
    'notes':           'str   — comentario del anotador, máximo 160 caracteres',
    # Metadatos del anotador
    'annotator_type':  '"human" | "llm"',
    'annotator_id':    'str   — iniciales, máx. 3 caracteres (ej. "AKM", "G4O")',
    'annotator_model': 'str   — nombre del modelo LLM (ej. "gpt-4o"), null si humano',
    'skill_file':      'str   — skill .md usado, null si humano',
    'score_confianza': 'float — 0.0–1.0 para LLM, null para humano',
    'justificacion':   'str   — justificación textual del LLM, vacío para humano',
    # Auditoría
    'annotated_at':    'str   — ISO 8601 timestamp',
}

print('Esquema de registro de etiquetado:')
for campo, desc in ANNOTATED_FIELDS.items():
    print(f'  {campo:<20}  {desc}')


## 4. Consolidación multi-anotador

El mismo chunk puede ser anotado por múltiples humanos y/o LLMs. El JSONL de salida tiene **una fila por `(chunk_id, annotator_id)`**.

| Escenario | Uso |
|---|---|
| Humano + LLM | El humano revisa y confirma/corrige la etiqueta del LLM |
| Múltiples humanos | Calcular Cohen's kappa por categoría antes del entrenamiento |
| Múltiples LLMs | Comparar acuerdo entre modelos; detectar sesgos |

`consolidar_anotaciones` produce el *gold standard* por mayoría de votos (>50%) entre todos los anotadores de cada chunk. El resultado continúa siendo ligero. Para guardarlo y usarlo en el cuaderno 04: `consolidar_anotaciones(output_path=CONSENSUS_FILE)`. Cuando se necesite inspeccionar el contenido, `vincular_anotaciones_con_chunks()` recupera texto y contexto desde el JSONL original mediante `chunk_id`.

In [6]:
LABELED_FILE = LABELED_DIR / 'labeled_chunks.jsonl'
CHUNKS_FILE = PROCESSED_DIR / 'chunks_para_etiquetar.jsonl'
CONSENSUS_FILE = PROCESSED_DIR / 'dataset_etiquetado.jsonl'
ANNOTATION_EXPORT_FIELDS = tuple(ANNOTATED_FIELDS)


def _load_jsonl(path):
    if not path.exists():
        return []
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def _write_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def compact_annotation(record):
    """Normaliza una anotación y elimina copias de texto/metadatos del chunk."""
    compact = {field: record.get(field) for field in ANNOTATION_EXPORT_FIELDS}
    compact['labels'] = compact.get('labels') if isinstance(compact.get('labels'), list) else []
    compact['flags'] = compact.get('flags') if isinstance(compact.get('flags'), list) else []
    compact['needs_review'] = bool(compact.get('needs_review'))
    compact['notes'] = str(compact.get('notes') or '').strip()[:160]
    compact['justificacion'] = str(compact.get('justificacion') or '').strip()[:500]
    if record.get('n_anotadores') is not None:
        compact['n_anotadores'] = int(record['n_anotadores'])
    return compact


def validar_anotacion(record):
    """Aplica el mismo contrato semántico usado por los cuadernos LLM."""
    errors = []
    labels_value = record.get('labels')
    flags_value = record.get('flags')
    if not isinstance(labels_value, list) or not labels_value:
        errors.append('labels debe ser una lista no vacía')
        labels_set = set()
    else:
        labels_set = set(labels_value)
        if len(labels_set) != len(labels_value):
            errors.append('labels no admite duplicados')
        if not labels_set <= ALLOWED_LABELS:
            errors.append('labels contiene valores fuera de la taxonomía')
    if not isinstance(flags_value, list):
        errors.append('flags debe ser una lista')
        flags_set = set()
    else:
        flags_set = set(flags_value)
        if len(flags_set) != len(flags_value):
            errors.append('flags no admite duplicados')
        if not flags_set <= ALLOWED_FLAGS:
            errors.append('flags contiene valores fuera de la taxonomía')
    safe = labels_set & SAFE_LABELS
    damage = labels_set & DAMAGE_LABELS
    if safe and damage:
        errors.append('una etiqueta segura no puede coexistir con daño')
    if len(safe) > 1:
        errors.append('las dos etiquetas seguras no pueden coexistir')
    if flags_set and not damage:
        errors.append('los flags requieren al menos una etiqueta de daño')
    if flags_set and record.get('needs_review') is not True:
        errors.append('cualquier flag obliga needs_review=true')
    score = record.get('score_confianza')
    if score is not None:
        if isinstance(score, bool) or not isinstance(score, (int, float)) or not 0 <= score <= 1:
            errors.append('score_confianza debe ser null o estar entre 0 y 1')
        elif ({'ironia_ambigua', 'contexto_necesario'} & flags_set) and score > 0.65:
            errors.append('flag ambiguo/contextual limita score_confianza a 0.65')
        elif score < 0.70 and record.get('needs_review') is not True:
            errors.append('score < 0.70 obliga needs_review=true')
    if not isinstance(record.get('notes'), str):
        errors.append('notes debe ser texto')
    if not isinstance(record.get('justificacion'), str):
        errors.append('justificacion debe ser texto')
    annotator_type = record.get('annotator_type')
    annotator_id = record.get('annotator_id')
    if annotator_type not in {'human', 'llm', 'consensus'}:
        errors.append('annotator_type debe ser human, llm o consensus')
    if not isinstance(annotator_id, str) or not annotator_id or len(annotator_id) > 3:
        errors.append('annotator_id debe tener entre 1 y 3 caracteres')
    if annotator_type == 'llm':
        if not isinstance(annotator_id, str) or not re.fullmatch(r'[A-Z0-9]{3}', annotator_id):
            errors.append('annotator_id LLM debe tener exactamente tres caracteres A-Z/0-9')
        if not str(record.get('annotator_model') or '').strip():
            errors.append('una anotación LLM requiere annotator_model')
        if not str(record.get('skill_file') or '').strip():
            errors.append('una anotación LLM requiere skill_file')
        if score is None:
            errors.append('una anotación LLM requiere score_confianza')
        if not str(record.get('justificacion') or '').strip():
            errors.append('una anotación LLM requiere justificacion')
    elif score is not None:
        errors.append('score_confianza debe ser null para humano/consenso')
    return errors


def load_annotation_rows(source=LABELED_DIR):
    """Lee un JSONL o todos los *_labeled_chunks.jsonl de un directorio."""
    source = Path(source)
    if source.is_dir():
        paths = sorted(source.glob('*_labeled_chunks.jsonl'))
        if not paths and (source / 'labeled_chunks.jsonl').exists():
            paths = [source / 'labeled_chunks.jsonl']
    else:
        paths = [source] if source.exists() else []
    rows = []
    for path in paths:
        for line_number, raw in enumerate(_load_jsonl(path), 1):
            row = compact_annotation(raw)
            errors = validar_anotacion(row)
            if errors:
                raise ValueError(f'Anotación inválida en {path}, fila {line_number}: {errors}')
            rows.append(row)
    keys = [(row.get('chunk_id'), row.get('annotator_id')) for row in rows]
    if any(not chunk_id or not annotator_id for chunk_id, annotator_id in keys):
        raise ValueError('Todas las anotaciones requieren chunk_id y annotator_id.')
    if len(keys) != len(set(keys)):
        raise ValueError('Hay anotaciones duplicadas para (chunk_id, annotator_id).')
    return rows


def vincular_anotaciones_con_chunks(source=LABELED_DIR, chunks_path=CHUNKS_FILE):
    """Recupera texto y contexto canónicos mediante un cruce many-to-one por chunk_id."""
    annotations = pd.DataFrame(load_annotation_rows(source))
    chunks = pd.DataFrame(_load_jsonl(Path(chunks_path)))
    if annotations.empty:
        return annotations
    if 'chunk_id' not in chunks or 'text' not in chunks:
        raise ValueError('El dataset original debe contener chunk_id y text.')
    if chunks['chunk_id'].isna().any() or chunks['chunk_id'].duplicated().any():
        raise ValueError('chunk_id debe ser único y no nulo en el dataset original.')
    if annotations['chunk_id'].isna().any():
        raise ValueError('Hay anotaciones sin chunk_id.')
    missing = sorted(set(annotations['chunk_id']) - set(chunks['chunk_id']))
    if missing:
        preview = ', '.join(missing[:5])
        raise ValueError(f'{len(missing)} chunk_id no existen en el dataset original: {preview}')
    source_columns = ['chunk_id'] + [c for c in chunks.columns if c not in annotations.columns]
    return annotations.merge(chunks[source_columns], on='chunk_id', how='left', validate='many_to_one')


def save_annotation(record, path=LABELED_FILE):
    """Guarda o actualiza una anotación. Clave única: (chunk_id, annotator_id).
    Si ya existe un registro con la misma clave, lo reemplaza.
    """
    record = compact_annotation(record)
    if not record.get('chunk_id') or not record.get('annotator_id'):
        raise ValueError('chunk_id y annotator_id son obligatorios.')
    errors = validar_anotacion(record)
    if errors:
        raise ValueError(f'Anotación incompatible: {errors}')
    existing = _load_jsonl(path)
    key = (record.get('chunk_id'), record.get('annotator_id'))
    updated = [r for r in existing
               if (r.get('chunk_id'), r.get('annotator_id')) != key]
    updated.append(record)
    _write_jsonl((compact_annotation(r) for r in updated), path)


def consolidar_anotaciones(path=LABELED_DIR, min_anotadores=2, output_path=None):
    """Gold standard por mayoría de votos (>50%) entre anotadores del mismo chunk.

    Returns:
        DataFrame con una fila por chunk_id y etiquetas consensuadas.
        Columna 'n_anotadores' indica cuántos anotadores contribuyeron.
    """
    rows = load_annotation_rows(path)
    if not rows:
        print('Sin anotaciones para consolidar.')
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    consolidado = []
    sin_mayoria = 0
    for chunk_id, group in df.groupby('chunk_id'):
        n = len(group)
        if n < min_anotadores:
            continue
        label_counts = Counter(itertools.chain.from_iterable(group['labels'].dropna()))
        flag_counts  = Counter(itertools.chain.from_iterable(group['flags'].dropna()))
        row = compact_annotation(group.iloc[0].to_dict())
        row['labels']         = sorted([l for l, c in label_counts.items() if c / n > 0.5])
        row['flags']          = sorted([f for f, c in flag_counts.items()  if c / n > 0.5])
        if not row['labels']:
            sin_mayoria += 1
            continue
        if set(row['labels']) <= SAFE_LABELS:
            row['flags'] = []
        row['needs_review']   = bool(row['flags'])
        row['n_anotadores']   = n
        row['annotator_type'] = 'consensus'
        row['annotator_id']   = 'CON'
        row['annotator_model'] = None
        row['skill_file'] = None
        row['score_confianza'] = None
        row['justificacion'] = ''
        row['notes'] = ''
        row['annotated_at']   = pd.Timestamp.now().isoformat()
        errors = validar_anotacion(row)
        if errors:
            raise ValueError(f'Consenso incompatible para {chunk_id}: {errors}')
        consolidado.append(row)

    result = pd.DataFrame(consolidado)
    if output_path is not None:
        _write_jsonl(result.to_dict(orient='records'), output_path)
        print(f'Consenso ligero guardado en: {output_path}')
    if sin_mayoria:
        print(f'Chunks sin mayoría de etiqueta (pendientes de arbitraje): {sin_mayoria}')
    return result


def resumen_anotaciones(path=LABELED_DIR):
    """Muestra un resumen de las anotaciones disponibles por anotador."""
    rows = load_annotation_rows(path)
    if not rows:
        print(f'Sin anotaciones en {Path(path).name}')
        print(f'Deposita archivos *_labeled_chunks.jsonl en: {LABELED_DIR}')
        return
    df = pd.DataFrame(rows)
    print(f'Total de registros   : {len(df)}')
    print(f'Chunks únicos        : {df["chunk_id"].nunique()}')
    print(f'Anotadores únicos    : {df["annotator_id"].nunique()}')
    print()
    print(df.groupby(['annotator_type', 'annotator_id'])
            .size().rename('n_anotaciones').to_string())


resumen_anotaciones()


Sin anotaciones en etiquetado
Deposita archivos *_labeled_chunks.jsonl en: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado


In [7]:
import re
import json

def _load_chunks_for_embedding():
    jsonl_path = PROCESSED_DIR / 'chunks_para_etiquetar.jsonl'
    csv_path = PROCESSED_DIR / 'chunks_para_etiquetar.csv'

    if jsonl_path.exists():
        rows = _load_jsonl(jsonl_path)
        source = jsonl_path
    elif csv_path.exists():
        df = pd.read_csv(csv_path)
        rows = df.to_dict(orient='records')
        source = csv_path
    else:
        raise FileNotFoundError('No se encontró chunks_para_etiquetar.jsonl ni chunks_para_etiquetar.csv en datos/processed')

    keep_fields = [
        'chunk_id', 'video_id', 'channel_id', 'channel_title', 'video_title',
        'published_at', 'start_seconds', 'end_seconds', 'text', 'text_hash'
    ]
    cleaned = []
    for row in rows:
        out = {k: row.get(k) for k in keep_fields}
        out['labels'] = row.get('labels') or []
        out['flags'] = row.get('flags') or []
        out['needs_review'] = bool(row.get('needs_review', False))
        out['notes'] = row.get('notes') or ''
        cleaned.append(out)
    return cleaned, source


def embed_chunks_in_html(html_file=FRONTEND_DIR / 'etiquetado_humano.html'):
    rows, source = _load_chunks_for_embedding()
    json_payload = json.dumps(rows, ensure_ascii=False)

    html = html_file.read_text(encoding='utf-8')
    pattern = r"<script id='embeddedChunks' type='application/json'>[\s\S]*?</script>"
    replacement = f"<script id='embeddedChunks' type='application/json'>{json_payload}</script>"
    if not re.search(pattern, html):
        raise ValueError("No se encontró el bloque <script id='embeddedChunks'> en el HTML")

    html_updated = re.sub(pattern, replacement, html, count=1)
    html_file.write_text(html_updated, encoding='utf-8')

    print(f'Fuente de chunks : {source}')
    print(f'Chunks embebidos : {len(rows)}')
    print(f'HTML actualizado : {html_file}')


embed_chunks_in_html()

Fuente de chunks : D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\processed\chunks_para_etiquetar.jsonl
Chunks embebidos : 69853
HTML actualizado : D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\Cuadernos\frontend\etiquetado_humano.html


In [ ]:
import ast
from pathlib import Path

LABEL_HELP = {
    'SEGURO': {
        'seguro': 'Sin infraccion: texto informativo, descriptivo o humor sin ataque a persona o grupo.',
        'seguro_ironia_marcada': 'Parodia/ironia cuyo blanco NO es un grupo humano.',
    },
    'RACISMO_DISCRIMINACION': {
        'racismo_etnico_explicito': 'Uso derogatorio de terminos etnicos (ej. serrano/cholo/negro/indio).',
        'racismo_linguistico': 'Burla de acento andino, motoseo u ortografia de migrantes.',
        'clasismo_racial': 'Inferiorizacion por clase con connotacion etnica (ej. huachafa/chusma/amixer).',
        'discriminacion_regional': 'Ataque por origen regional (Lima vs provincias, etc.).',
        'racismo_encubierto': 'Discriminacion disfrazada de criterio de cultura/educacion.',
    },
    'ACOSO': {
        'misoginia_acoso_genero': 'Insultos sexualizados, degradacion o ataque por genero.',
        'homofobia_transfobia': 'Insultos o amenazas contra personas LGBTQ+.',
        'acoso_personal': 'Ataque dirigido a persona identificable o doxeo.',
        'amenaza_directa': 'Expresion explicita de intencion de dano fisico/legal/economico.',
    },
    'CONTENIDO_SEXUAL': {
        'sexual_explicito': 'Descripcion grafica de actos sexuales sin proposito informativo.',
        'sexual_cosificacion': 'Sexualizacion/cosificacion de personas.',
        'sexual_no_consensual': 'Referencia a contenido sexual no consensual.',
    },
}

FLAG_HELP = {
    'ironia_ambigua': 'No se determina claramente si hay dano o parodia critica. Requiere revision.',
    'humor_encubridor': 'El humor funciona como cobertura para minimizar/normalizar el dano.',
    'contexto_necesario': 'El chunk aislado no alcanza; se necesita contexto adicional del video.',
}

def _coerce_list(value):
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
    return []

def load_chunks_dataset(dataset_path=None):
    """Carga chunks desde .jsonl, .json o .csv para incrustarlos en el frontend."""
    if dataset_path is None:
        default_jsonl = PROCESSED_DIR / 'chunks_para_etiquetar.jsonl'
        default_csv = PROCESSED_DIR / 'chunks_para_etiquetar.csv'
        dataset_path = default_jsonl if default_jsonl.exists() else default_csv

    path = Path(dataset_path)
    if not path.exists():
        raise FileNotFoundError(f'No existe dataset: {path}')

    suffix = path.suffix.lower()
    if suffix == '.jsonl':
        rows = _load_jsonl(path)
    elif suffix == '.json':
        rows = json.loads(path.read_text(encoding='utf-8'))
        if not isinstance(rows, list):
            raise ValueError('El .json debe contener una lista de registros.')
    elif suffix == '.csv':
        rows = pd.read_csv(path).to_dict(orient='records')
    else:
        raise ValueError('Formato no soportado. Usa .jsonl, .json o .csv')

    keep_fields = [
        'chunk_id', 'video_id', 'channel_id', 'channel_title', 'video_title',
        'published_at', 'start_seconds', 'end_seconds', 'text', 'text_hash'
    ]
    cleaned = []
    for row in rows:
        out = {k: row.get(k) for k in keep_fields}
        out['labels'] = _coerce_list(row.get('labels'))
        out['flags'] = _coerce_list(row.get('flags'))
        out['needs_review'] = bool(row.get('needs_review', False))
        out['notes'] = row.get('notes') or ''
        cleaned.append(out)

    return cleaned, path

def write_labels_guide_html(output_path=FRONTEND_DIR / 'guia_etiquetas.html'):
    """Genera la segunda pagina con explicacion de etiquetas y flags del proyecto."""
    output_path = Path(output_path)
    cards = []
    for category, labels in LABEL_HELP.items():
        items = ''.join([f"<li><strong>{k}</strong>: {v}</li>" for k, v in labels.items()])
        cards.append(f"<section class='card'><h2>{category}</h2><ul>{items}</ul></section>")

    flag_items = ''.join([f"<li><strong>{k}</strong>: {v}</li>" for k, v in FLAG_HELP.items()])

    html = f"""<!doctype html>
<html lang='es'>
<head>
  <meta charset='utf-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1'>
  <title>Guia de etiquetas - Moderacion</title>
  <style>
    :root {{ --bg:#f6f7f9; --panel:#fff; --ink:#18202b; --muted:#5d6b7a; --line:#d7dde5; --brand:#b4232f; }}
    * {{ box-sizing: border-box; }}
    body {{ margin:0; font-family: Arial, Helvetica, sans-serif; background:var(--bg); color:var(--ink); }}
    header {{ background:var(--brand); color:#fff; padding:14px 22px; }}
    main {{ max-width:1200px; margin:0 auto; padding:18px; display:grid; gap:12px; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); }}
    .card {{ background:var(--panel); border:1px solid var(--line); border-radius:8px; padding:14px; }}
    h1 {{ margin:0; font-size:22px; }}
    h2 {{ margin:0 0 10px; font-size:16px; }}
    ul {{ margin:0; padding-left:18px; }}
    li {{ margin:6px 0; line-height:1.4; }}
    .note {{ max-width:1200px; margin:0 auto 14px; padding:0 18px; color:var(--muted); font-size:13px; }}
  </style>
</head>
<body>
  <header><h1>Guia de etiquetas del proyecto</h1></header>
  <p class='note'>Esta guia resume las etiquetas y flags definidos para el etiquetado humano/LLM en este proyecto.</p>
  <main>
    {''.join(cards)}
    <section class='card'>
      <h2>FLAGS TRANSVERSALES</h2>
      <ul>{flag_items}</ul>
    </section>
  </main>
</body>
</html>"""

    output_path.write_text(html, encoding='utf-8')
    return output_path

def build_frontend_html(rows=None, embed_data=True):
    """Construye el frontend con datos embebidos o con carga local de JSONL/JSON."""
    rows = rows or []
    json_payload = json.dumps(rows, ensure_ascii=False)
    data_source_block = (
        f"<script id='embeddedChunks' type='application/json'>{json_payload}</script>"
        if embed_data else ''
    )
    data_mode_help = (
        'Los chunks están incluidos en este archivo HTML.' if embed_data
        else 'Este HTML no contiene datos. Carga un archivo JSONL o JSON desde tu equipo.'
    )

    label_cards_html = []
    for category, labels in LABEL_HELP.items():
        items = ''.join([f"<li><strong>{k}</strong>: {v}</li>" for k, v in labels.items()])
        label_cards_html.append(f"<section class='guide-card'><h3>{category}</h3><ul>{items}</ul></section>")
    guide_labels_html = ''.join(label_cards_html)
    guide_flags_html = ''.join([f"<li><strong>{k}</strong>: {v}</li>" for k, v in FLAG_HELP.items()])

    html_template = """<!doctype html>
<html lang='es'>
<head>
  <meta charset='utf-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1'>
  <title>Etiquetado humano - Moderacion de contenido</title>
  <style>
    :root { --bg: #f6f7f9; --panel: #ffffff; --ink: #18202b; --muted: #5d6b7a; --line: #d7dde5; --brand: #b4232f; --ok: #0f7a45; --touch: 44px; }
    * { box-sizing: border-box; }
    body { margin: 0; font-family: Arial, Helvetica, sans-serif; background: var(--bg); color: var(--ink); }
    header { background: var(--brand); color: white; padding: 12px 16px; }
    main { max-width: 1320px; margin: 0 auto; padding: 14px; display: grid; grid-template-columns: 1fr 260px; gap: 14px; }
    section, aside { background: var(--panel); border: 1px solid var(--line); border-radius: 10px; padding: 14px; }
    h1 { font-size: 20px; margin: 0; }
    h2 { font-size: 16px; margin: 0 0 12px; }
    button { border: 1px solid var(--line); background: white; border-radius: 8px; padding: 10px 14px; cursor: pointer; min-height: var(--touch); font-size: 14px; }
    button.primary { background: var(--brand); color: white; border-color: var(--brand); }
    button.linklike { color: #1248a3; border-color: #bcd0ee; background: #f6f9ff; }
    .file-button { border: 1px solid #bcd0ee; background: #f6f9ff; color: #1248a3; border-radius: 8px; padding: 10px 14px; cursor: pointer; min-height: var(--touch); display: inline-flex; align-items: center; font-size: 14px; }
    .dataset-name { color: var(--muted); font-size: 12px; max-width: 240px; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
    input[type='text'], textarea { border: 1px solid var(--line); border-radius: 8px; font-family: inherit; }
    input[type='text'] { min-height: 40px; padding: 6px 10px; }
    textarea { width: 100%; min-height: 96px; resize: vertical; padding: 10px; }
    .toolbar { display: flex; gap: 8px; flex-wrap: wrap; align-items: center; margin-bottom: 10px; }
    .top-sticky { position: sticky; top: 0; z-index: 30; background: var(--panel); padding: 8px 0; border-bottom: 1px solid var(--line); }
    .meta { color: var(--muted); font-size: 13px; line-height: 1.35; margin-bottom: 10px; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
    .chunk { border: 1px solid var(--line); border-radius: 8px; padding: 14px; min-height: 220px; line-height: 1.55; background: #fbfcfd; }
    .neighbor-context { margin-top: 10px; border: 1px solid var(--line); border-radius: 8px; padding: 10px; background: #fffdf6; }
    .neighbor-context summary { cursor: pointer; font-size: 13px; font-weight: bold; }
    .neighbor-text { white-space: pre-wrap; font-size: 13px; color: var(--muted); margin: 8px 0; }
    .labels { display: grid; grid-template-columns: 1fr; gap: 8px; }
    .label { border: 1px solid var(--line); border-radius: 6px; padding: 8px; display: flex; gap: 8px; align-items: flex-start; cursor: pointer; }
    .label input { margin-top: 2px; transform: scale(1.1); }
    .label-text { font-size: 13px; user-select: none; }
    .annotation-layout { display: grid; grid-template-columns: 1fr 290px 290px; gap: 12px; align-items: start; }
    .labels-panel { border: 1px solid var(--line); border-radius: 8px; padding: 10px; background: #fcfdff; max-height: 74vh; overflow: auto; }
    .chunk-center { min-width: 0; }
    .video-link { margin-bottom: 10px; font-size: 13px; }
    .video-link a { color: #1248a3; text-decoration: none; font-weight: bold; }
    .video-link a:hover { text-decoration: underline; }
    .quick-links { margin-top: 8px; display: flex; flex-wrap: wrap; gap: 8px; }
    .status { margin-top: 12px; font-size: 13px; color: var(--muted); }
    .ok { color: var(--ok); font-weight: bold; }
    .cat-seguro { background: #eaf8ee; border-color: #b8e2c3; }
    .cat-racismo { background: #fff1f1; border-color: #f0c2c2; }
    .cat-acoso { background: #fff7e8; border-color: #efd7a9; }
    .cat-sexual { background: #f1edff; border-color: #d2c5ff; }
    .cat-flag { background: #eaf4ff; border-color: #bcd8f5; }

    .guide-dialog { width: min(980px, 96vw); border: none; border-radius: 12px; padding: 0; }
    .guide-dialog::backdrop { background: rgba(0, 0, 0, 0.45); }
    .guide-shell { background: #fff; color: var(--ink); }
    .guide-head { display: flex; justify-content: space-between; align-items: center; gap: 8px; padding: 12px 14px; border-bottom: 1px solid var(--line); position: sticky; top: 0; background: #fff; }
    .guide-grid { display: grid; gap: 10px; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr)); padding: 12px; }
    .guide-card { border: 1px solid var(--line); border-radius: 8px; padding: 10px; background: #fbfcfd; }
    .guide-card h3 { margin: 0 0 8px; font-size: 14px; }
    .guide-card ul { margin: 0; padding-left: 18px; }
    .guide-card li { margin: 6px 0; font-size: 13px; line-height: 1.4; }

    @media (max-width: 1150px) {
      main { grid-template-columns: 1fr; }
      .annotation-layout { grid-template-columns: 1fr; }
      .labels-panel { max-height: none; }
    }

    @media (max-width: 768px) {
      header { position: sticky; top: 0; z-index: 40; padding: 10px 12px; }
      h1 { font-size: 17px; }
      main { padding: 10px; gap: 10px; }
      section, aside { padding: 10px; border-radius: 8px; }
      .toolbar { gap: 6px; }
      .top-sticky { top: 0; margin: -4px 0 8px; }
      .toolbar button { flex: 1 1 31%; min-width: 96px; }
      #annotatorId, #annotatorModel, #skillFile { width: 100% !important; }
      .meta { font-size: 12px; }
      .chunk { min-height: 180px; padding: 12px; }
      .label { padding: 10px; }
      .label-text { font-size: 14px; }
      aside { display: none; }
    }
  </style>
</head>
<body>
  <header>
    <h1>Etiquetado humano - Moderacion de contenido</h1>
    <div class='quick-links'>
      <button type='button' class='linklike' data-open-guide='1'>Guia de etiquetas y ejemplos</button>
    </div>
  </header>
  <main>
    <section>
      <div class='toolbar top-sticky'>
        <strong style='font-size:12px'>Anotador</strong>
        <label style='font-size:12px'><input type='radio' name='ann_type' value='human' checked> Humano</label>
        <label style='font-size:12px'><input type='radio' name='ann_type' value='llm'> LLM</label>
        <input id='annotatorId' type='text' maxlength='3' placeholder='Iniciales (max. 3)' style='width:140px;font-size:12px'>
        <input id='annotatorModel' type='text' placeholder='Modelo LLM (ej. gpt-4o)' style='display:none;width:190px;font-size:12px'>
        <input id='skillFile' type='text' value='clasificacion_moderacion_peru.md' style='display:none;width:220px;font-size:12px;color:var(--muted)'>
      </div>
      <div class='toolbar'>
        <button id='prevBtn'>Anterior</button>
        <button id='nextBtn' class='primary'>Siguiente</button>
        <button id='exportBtn'>Exportar JSONL</button>
        <label for='datasetFile' class='file-button'>Cargar JSONL/JSON</label>
        <input id='datasetFile' type='file' accept='.jsonl,.json,application/json' hidden>
        <span id='datasetName' class='dataset-name'></span>
        <button id='openGuideBtn' class='linklike' type='button' data-open-guide='1'>Guia</button>
      </div>
      <div id='meta' class='meta'>Esperando datos...</div>
      <div id='videoSource' class='video-link'>Video original: cargando...</div>
      <div class='annotation-layout'>
        <div class='chunk-center'>
          <div id='chunkText' class='chunk'>Preparando interfaz...</div>
          <details id='neighborContext' class='neighbor-context'>
            <summary>Contexto vecino del mismo video (opcional)</summary>
            <div><strong>Chunk anterior</strong><div id='previousContext' class='neighbor-text'></div></div>
            <div><strong>Chunk posterior</strong><div id='nextContext' class='neighbor-text'></div></div>
          </details>
          <h2 style='margin-top:14px'>Notas</h2>
          <textarea id='notes' maxlength='160' placeholder='Comentario breve del anotador (máx. 160 caracteres)'></textarea>
          <div id='llmSemanticFields' style='display:none'>
            <h2 style='margin-top:14px'>Datos de la decisión LLM</h2>
            <label style='font-size:12px'>Confianza (0–1) <input id='scoreConfianza' type='number' min='0' max='1' step='0.01' style='width:100px'></label>
            <textarea id='justificacion' maxlength='500' placeholder='Justificación breve y concreta (máx. 500 caracteres)'></textarea>
          </div>
          <div id='status' class='status'></div>
        </div>
        <div id='labelsLeft' class='labels-panel labels'></div>
        <div id='labelsRight' class='labels-panel labels'></div>
      </div>
    </section>
    <aside>
      <h2>Ayuda rapida</h2>
      <p style='font-size:13px;color:var(--muted);line-height:1.45'>__DATA_MODE_HELP__</p>
      <button type='button' data-open-guide='1' class='linklike'>Abrir guia de etiquetas</button>
    </aside>
  </main>

  <dialog id='guideDialog' class='guide-dialog'>
    <div class='guide-shell'>
      <div class='guide-head'>
        <h2 style='margin:0'>Guia de etiquetas del proyecto</h2>
        <button type='button' id='closeGuideBtn'>Cerrar</button>
      </div>
      <div class='guide-grid'>
        __GUIDE_LABEL_CARDS__
        <section class='guide-card'>
          <h3>FLAGS TRANSVERSALES</h3>
          <ul>__GUIDE_FLAG_ITEMS__</ul>
        </section>
      </div>
    </div>
  </dialog>

  __DATA_SOURCE_BLOCK__
  <script>
    const LABEL_SECTIONS = [
      ['SEGURO', ['seguro', 'seguro_ironia_marcada']],
      ['RACISMO / DISCRIMINACION', ['racismo_etnico_explicito','racismo_linguistico','clasismo_racial','discriminacion_regional','racismo_encubierto']],
      ['ACOSO', ['misoginia_acoso_genero','homofobia_transfobia','acoso_personal','amenaza_directa']],
      ['CONTENIDO SEXUAL', ['sexual_explicito','sexual_cosificacion','sexual_no_consensual']],
    ];
    const SECTION_CLASS = {
      'SEGURO': 'cat-seguro',
      'RACISMO / DISCRIMINACION': 'cat-racismo',
      'ACOSO': 'cat-acoso',
      'CONTENIDO SEXUAL': 'cat-sexual',
    };
    const SAFE_LABELS = ['seguro', 'seguro_ironia_marcada'];
    const DAMAGE_LABELS = LABEL_SECTIONS.flatMap(([, labels]) => labels).filter(label => !SAFE_LABELS.includes(label));
    const FLAGS = ['ironia_ambigua', 'humor_encubridor', 'contexto_necesario'];
    const DISPLAY_LABELS = {
      acoso_personal: 'Acoso personal/lenguaje obceno',
      amenaza_directa: 'Amenza directa/Violencia'
    };
    const state = { rows: [], order: [], index: 0, neighbors: {} };
    const meta = document.getElementById('meta');
    const chunkText = document.getElementById('chunkText');
    const labelsLeft = document.getElementById('labelsLeft');
    const labelsRight = document.getElementById('labelsRight');
    const notes = document.getElementById('notes');
    const scoreConfianza = document.getElementById('scoreConfianza');
    const justificacion = document.getElementById('justificacion');
    const llmSemanticFields = document.getElementById('llmSemanticFields');
    const status = document.getElementById('status');
    const videoSource = document.getElementById('videoSource');
    const neighborContext = document.getElementById('neighborContext');
    const previousContext = document.getElementById('previousContext');
    const nextContext = document.getElementById('nextContext');
    const STORAGE_KEY = 'moderacion.annotator.v1';
    const guideDialog = document.getElementById('guideDialog');
    const datasetFile = document.getElementById('datasetFile');
    const datasetName = document.getElementById('datasetName');

    function shuffleInPlace(arr) {
      for (let i = arr.length - 1; i > 0; i -= 1) {
        const j = Math.floor(Math.random() * (i + 1));
        [arr[i], arr[j]] = [arr[j], arr[i]];
      }
      return arr;
    }

    function currentRow() {
      if (!state.rows.length || !state.order.length) return null;
      return state.rows[state.order[state.index]] || null;
    }

    function buildNeighborMap(rows) {
      const groups = new Map();
      rows.forEach((row, sourceIndex) => {
        const videoId = String(row.video_id || '').trim();
        if (!videoId) return;
        if (!groups.has(videoId)) groups.set(videoId, []);
        groups.get(videoId).push({ row, sourceIndex });
      });
      const neighbors = {};
      groups.forEach(items => {
        items.sort((a, b) => (Number(a.row.start_seconds) || 0) - (Number(b.row.start_seconds) || 0) || a.sourceIndex - b.sourceIndex);
        items.forEach((item, i) => {
          neighbors[item.row.chunk_id] = {
            previous: i > 0 ? items[i - 1].row.text || '' : '',
            next: i + 1 < items.length ? items[i + 1].row.text || '' : '',
          };
        });
      });
      return neighbors;
    }

    function formatLabelName(label) {
      return DISPLAY_LABELS[label] || label.replaceAll('_', ' ');
    }

    function createSectionBlock(parent, sectionName, labels) {
      const hdr = document.createElement('div');
      hdr.style.cssText = 'font-size:10px;font-weight:bold;color:var(--muted);text-transform:uppercase;margin:6px 0 2px;letter-spacing:.05em';
      hdr.textContent = sectionName;
      parent.appendChild(hdr);
      const sectionClass = SECTION_CLASS[sectionName] || '';
      labels.forEach(label => {
        const wrap = document.createElement('label');
        wrap.className = `label ${sectionClass}`;
        wrap.innerHTML = `<input class='label-checkbox' type='checkbox' value='${label}'> <span class='label-text'>${formatLabelName(label)}</span>`;
        parent.appendChild(wrap);
      });
    }

    function renderLabels() {
      labelsLeft.innerHTML = '';
      labelsRight.innerHTML = '';
      const leftSections = LABEL_SECTIONS.slice(0, 2);
      const rightSections = LABEL_SECTIONS.slice(2);
      leftSections.forEach(([name, labels]) => createSectionBlock(labelsLeft, name, labels));
      rightSections.forEach(([name, labels]) => createSectionBlock(labelsRight, name, labels));

      const fhdr = document.createElement('div');
      fhdr.style.cssText = 'font-size:10px;font-weight:bold;color:var(--muted);text-transform:uppercase;margin:8px 0 2px;letter-spacing:.05em;border-top:1px solid var(--line);padding-top:6px';
      fhdr.textContent = 'FLAGS TRANSVERSALES';
      labelsRight.appendChild(fhdr);
      FLAGS.forEach(flag => {
        const wrap = document.createElement('label');
        wrap.className = 'label cat-flag';
        wrap.innerHTML = `<input class='label-checkbox' type='checkbox' value='${flag}'> <span class='label-text'><em>${formatLabelName(flag)}</em></span>`;
        labelsRight.appendChild(wrap);
      });

      const labelsRoot = document.querySelector('.annotation-layout');
      labelsRoot.addEventListener('click', event => {
        if (!event.target.classList.contains('label-text')) return;
        event.preventDefault();
        const label = event.target.closest('label');
        if (!label) return;
        const checkbox = label.querySelector('.label-checkbox');
        if (!checkbox) return;
        checkbox.checked = !checkbox.checked;
        checkbox.dispatchEvent(new Event('change', { bubbles: true }));
      });

      labelsRoot.addEventListener('change', event => {
        if (!event.target.classList.contains('label-checkbox')) return;
        const val = event.target.value;
        if (SAFE_LABELS.includes(val) && event.target.checked) {
          document.querySelectorAll('.label-checkbox').forEach(x => {
            if (x.value !== val) x.checked = false;
          });
        }
        if (DAMAGE_LABELS.includes(val) && event.target.checked) {
          SAFE_LABELS.forEach(s => {
            const el = document.querySelector(`.label-checkbox[value='${s}']`);
            if (el) el.checked = false;
          });
        }
        if (FLAGS.includes(val) && event.target.checked) {
          const hasDamage = [...document.querySelectorAll('.label-checkbox:checked')]
            .some(x => DAMAGE_LABELS.includes(x.value));
          if (!hasDamage) {
            event.target.checked = false;
            alert('Los flags solo pueden acompañar al menos una etiqueta de daño.');
          }
        }
        const hasDamage = [...document.querySelectorAll('.label-checkbox:checked')]
          .some(x => DAMAGE_LABELS.includes(x.value));
        if (!hasDamage) {
          FLAGS.forEach(flag => {
            const el = document.querySelector(`.label-checkbox[value='${flag}']`);
            if (el) el.checked = false;
          });
        }
        saveCurrent();
      });
    }

    function parseRows(text) {
      const trimmed = text.trim();
      if (!trimmed) return [];
      if (trimmed.startsWith('[')) return JSON.parse(trimmed);
      return trimmed.split(String.fromCharCode(10)).map(line => line.replace(String.fromCharCode(13), '')).filter(Boolean).map(line => JSON.parse(line));
    }

    function selectedLabels() {
      return [...document.querySelectorAll('.label-checkbox:checked')].map(x => x.value);
    }

    function formatTimestamp(totalSeconds) {
      const sec = Math.max(0, Math.floor(Number(totalSeconds) || 0));
      const h = Math.floor(sec / 3600);
      const m = Math.floor((sec % 3600) / 60);
      const s = sec % 60;
      if (h > 0) return `${h}:${String(m).padStart(2, '0')}:${String(s).padStart(2, '0')}`;
      return `${String(m).padStart(2, '0')}:${String(s).padStart(2, '0')}`;
    }

    function getAnnotatorMeta() {
      const type = document.querySelector('input[name=ann_type]:checked')?.value || 'human';
      const id = (document.getElementById('annotatorId')?.value || '').trim().toUpperCase().slice(0, 3);
      const model = type === 'llm' ? (document.getElementById('annotatorModel')?.value.trim() || null) : null;
      const skill = type === 'llm' ? (document.getElementById('skillFile')?.value.trim() || null) : null;
      return { annotator_type: type, annotator_id: id, annotator_model: model, skill_file: skill };
    }

    function persistAnnotatorMeta() {
      localStorage.setItem(STORAGE_KEY, JSON.stringify(getAnnotatorMeta()));
    }

    function restoreAnnotatorMeta() {
      try {
        const raw = localStorage.getItem(STORAGE_KEY);
        if (!raw) return;
        const ann = JSON.parse(raw);
        const radio = document.querySelector(`input[name=ann_type][value='${ann.annotator_type || 'human'}']`);
        if (radio) radio.checked = true;
        if (ann.annotator_id) document.getElementById('annotatorId').value = ann.annotator_id;
        if (ann.annotator_model) document.getElementById('annotatorModel').value = ann.annotator_model;
        if (ann.skill_file) document.getElementById('skillFile').value = ann.skill_file;
      } catch (e) {}
    }

    function updateAnnotatorFieldVisibility() {
      const llm = document.querySelector('input[name=ann_type]:checked')?.value === 'llm';
      document.getElementById('annotatorModel').style.display = llm ? 'inline-block' : 'none';
      document.getElementById('skillFile').style.display = llm ? 'inline-block' : 'none';
      llmSemanticFields.style.display = llm ? 'block' : 'none';
    }

    function validateAnnotationState(row) {
      const errors = [];
      const rowLabels = Array.isArray(row.labels) ? row.labels : [];
      const rowFlags = Array.isArray(row.flags) ? row.flags : [];
      const safe = rowLabels.filter(label => SAFE_LABELS.includes(label));
      const damage = rowLabels.filter(label => DAMAGE_LABELS.includes(label));
      if (!rowLabels.length) errors.push('falta una etiqueta principal');
      if (new Set(rowLabels).size !== rowLabels.length) errors.push('hay etiquetas duplicadas');
      if (rowLabels.some(label => !SAFE_LABELS.includes(label) && !DAMAGE_LABELS.includes(label))) errors.push('hay etiquetas fuera de la taxonomía');
      if (new Set(rowFlags).size !== rowFlags.length) errors.push('hay flags duplicados');
      if (rowFlags.some(flag => !FLAGS.includes(flag))) errors.push('hay flags fuera de la taxonomía');
      if (safe.length > 1) errors.push('las dos etiquetas seguras no pueden coexistir');
      if (safe.length && damage.length) errors.push('seguro no puede coexistir con daño');
      if (rowFlags.length && !damage.length) errors.push('los flags requieren una etiqueta de daño');
      if (rowFlags.length && row.needs_review !== true) errors.push('los flags requieren revisión');
      if ((row.notes || '').length > 160) errors.push('notes supera 160 caracteres');
      const annotatorType = row.annotator_type || 'human';
      const annotatorId = row.annotator_id || '';
      if (!['human', 'llm', 'consensus'].includes(annotatorType)) errors.push('tipo de anotador inválido');
      if (!annotatorId || annotatorId.length > 3) errors.push('ID de anotador inválido');
      if (annotatorType === 'llm') {
        const score = row.score_confianza;
        if (!/^[A-Z0-9]{3}$/.test(annotatorId)) errors.push('ID LLM debe tener tres caracteres A-Z/0-9');
        if (!(row.annotator_model || '').trim()) errors.push('falta el modelo LLM');
        if (!(row.skill_file || '').trim()) errors.push('falta el skill del LLM');
        if (typeof score !== 'number' || !Number.isFinite(score) || score < 0 || score > 1) {
          errors.push('la confianza LLM debe estar entre 0 y 1');
        }
        if (!(row.justificacion || '').trim()) errors.push('falta justificación LLM');
        if ((row.justificacion || '').length > 500) errors.push('justificación supera 500 caracteres');
        if (rowFlags.some(flag => ['ironia_ambigua', 'contexto_necesario'].includes(flag)) && score > 0.65) {
          errors.push('ironía/contexto limita la confianza a 0.65');
        }
        if (typeof score === 'number' && score < 0.70 && row.needs_review !== true) {
          errors.push('confianza menor de 0.70 requiere revisión');
        }
      } else if (row.score_confianza != null) {
        errors.push('la confianza debe ser nula para humano/consenso');
      }
      return errors;
    }

    function isTouched(row) {
      return (row.labels || []).length > 0 || (row.flags || []).length > 0
        || Boolean((row.notes || '').trim()) || row.score_confianza != null
        || Boolean((row.justificacion || '').trim());
    }

    function isLabeled(row) {
      return isTouched(row) && validateAnnotationState(row).length === 0;
    }

    function saveCurrent() {
      const row = currentRow();
      if (!row) return;
      const all = selectedLabels();
      const ann = getAnnotatorMeta();
      row.labels = all.filter(l => !FLAGS.includes(l));
      row.flags = all.filter(l => FLAGS.includes(l));
      row.notes = notes.value.trim().slice(0, 160);
      row.annotated_at = new Date().toISOString();
      row.annotator_type = ann.annotator_type;
      row.annotator_id = ann.annotator_id;
      row.annotator_model = ann.annotator_model;
      row.skill_file = ann.skill_file;
      if (ann.annotator_type === 'llm') {
        const parsedScore = scoreConfianza.value === '' ? null : Number(scoreConfianza.value);
        const contextual = row.flags.some(flag => ['ironia_ambigua', 'contexto_necesario'].includes(flag));
        row.score_confianza = contextual && parsedScore > 0.65 ? 0.65 : parsedScore;
        row.justificacion = justificacion.value.trim().slice(0, 500);
        if (row.score_confianza !== parsedScore && row.score_confianza != null) {
          scoreConfianza.value = String(row.score_confianza);
        }
      } else {
        row.score_confianza = null;
        row.justificacion = '';
      }
      row.needs_review = row.flags.length > 0
        || (typeof row.score_confianza === 'number' && row.score_confianza < 0.70);
      persistAnnotatorMeta();
    }

    function toAnnotationRecord(row) {
      return {
        chunk_id: row.chunk_id,
        labels: [...(row.labels || [])],
        flags: [...(row.flags || [])],
        needs_review: Boolean(row.needs_review),
        notes: row.notes || '',
        annotator_type: row.annotator_type || 'human',
        annotator_id: row.annotator_id || '',
        annotator_model: row.annotator_model ?? null,
        skill_file: row.skill_file ?? null,
        score_confianza: row.score_confianza ?? null,
        justificacion: row.justificacion || '',
        annotated_at: row.annotated_at || new Date().toISOString(),
      };
    }

    function buildMetaLine(row) {
      const idx = state.index + 1;
      const total = state.rows.length;
      const channel = row.channel_title || '';
      const title = row.video_title || row.title || '';
      const start = Number(row.start_seconds || 0).toFixed(1);
      const end = Number(row.end_seconds || 0).toFixed(1);
      if (window.innerWidth <= 768) {
        return `${idx}/${total} | ${channel || 'Canal'} | ${start}s-${end}s`;
      }
      return `${idx}/${total} (orden aleatorio) | ${channel} | ${title} | ${start}s - ${end}s`;
    }

    function render() {
      const row = currentRow();
      if (!row) {
        meta.textContent = 'Sin chunks cargados.';
        chunkText.textContent = 'Selecciona Cargar JSONL/JSON y elige el dataset de chunks.';
        videoSource.textContent = 'Video original: no disponible';
        neighborContext.hidden = true;
        return;
      }
      meta.textContent = buildMetaLine(row);
      const start = Number(row.start_seconds || 0);
      const videoId = (row.video_id || '').trim();
      if (videoId) {
        const ts = formatTimestamp(start);
        const url = `https://www.youtube.com/watch?v=${videoId}&t=${Math.floor(start)}s`;
        videoSource.innerHTML = `Video original en YouTube: <a href='${url}' target='_blank' rel='noopener'>${ts}</a>`;
      } else {
        videoSource.textContent = 'Video original: no disponible para este chunk';
      }
      chunkText.textContent = row.text || '';
      const derivedContext = state.neighbors[row.chunk_id] || {};
      const previousText = row.contexto_anterior || derivedContext.previous || '';
      const nextText = row.contexto_posterior || derivedContext.next || '';
      neighborContext.hidden = !previousText && !nextText;
      previousContext.textContent = previousText || 'No hay chunk anterior disponible.';
      nextContext.textContent = nextText || 'No hay chunk posterior disponible.';
      const active = [...(row.labels || []), ...(row.flags || [])];
      document.querySelectorAll('.label-checkbox').forEach(x => x.checked = active.includes(x.value));
      notes.value = row.notes || '';
      scoreConfianza.value = row.score_confianza ?? '';
      justificacion.value = row.justificacion || '';
      updateAnnotatorFieldVisibility();
      const ann = getAnnotatorMeta();
      const etiquetados = state.rows.filter(isLabeled).length;
      const icono = ann.annotator_type === 'llm' ? '🤖' : '👤';
      status.innerHTML = `<span class='ok'>${etiquetados}</span> etiquetados de ${state.rows.length} | ${icono} <strong>${ann.annotator_id || '—'}</strong>`;
    }

    function loadEmbeddedChunks() {
      const source = document.getElementById('embeddedChunks');
      if (!source) return [];
      const text = source.textContent || '[]';
      const parsed = parseRows(text);
      return Array.isArray(parsed) ? parsed : [];
    }

    function activateRows(rows, sourceName = '') {
      if (!Array.isArray(rows)) throw new Error('El dataset debe ser una lista JSON o un archivo JSONL.');
      const seen = new Set();
      rows.forEach((row, index) => {
        const chunkId = String(row?.chunk_id || '').trim();
        if (!chunkId) throw new Error(`Registro ${index + 1}: falta chunk_id.`);
        if (seen.has(chunkId)) throw new Error(`chunk_id duplicado: ${chunkId}`);
        if (typeof row.text !== 'string') throw new Error(`Chunk ${chunkId}: falta el texto canónico.`);
        seen.add(chunkId);
      });
      state.rows = rows;
      state.neighbors = buildNeighborMap(rows);
      state.order = shuffleInPlace(rows.map((_, i) => i));
      state.index = 0;
      datasetName.textContent = sourceName;
      render();
    }

    async function loadDatasetFile(file) {
      if (!file) return;
      try {
        const rows = parseRows(await file.text());
        activateRows(rows, `${file.name} · ${rows.length} chunks`);
      } catch (error) {
        activateRows([], 'Error de carga');
        alert(`No se pudo cargar el dataset: ${error.message}`);
      }
    }

    function bindGuideDialog() {
      if (!guideDialog) return;
      document.querySelectorAll('[data-open-guide="1"]').forEach(btn => {
        btn.addEventListener('click', () => {
          if (typeof guideDialog.showModal === 'function') {
            guideDialog.showModal();
          }
        });
      });
      const closeBtn = document.getElementById('closeGuideBtn');
      if (closeBtn) closeBtn.addEventListener('click', () => guideDialog.close());
      guideDialog.addEventListener('click', (event) => {
        if (event.target === guideDialog) guideDialog.close();
      });
    }

    document.getElementById('prevBtn').addEventListener('click', () => { saveCurrent(); state.index = Math.max(0, state.index - 1); render(); });
    document.getElementById('nextBtn').addEventListener('click', () => { saveCurrent(); state.index = Math.min(state.rows.length - 1, state.index + 1); render(); });
    notes.addEventListener('input', saveCurrent);
    document.getElementById('exportBtn').addEventListener('click', () => {
      saveCurrent();
      const ann = getAnnotatorMeta();
      if (!ann.annotator_id) {
        alert('Configura las iniciales del anotador antes de exportar.');
        return;
      }
      const prefix = ann.annotator_id ? ann.annotator_id.toLowerCase() + '_' : '';
      const touchedRows = state.rows.filter(isTouched);
      const invalidRows = touchedRows
        .map(row => ({ row, errors: validateAnnotationState(row) }))
        .filter(item => item.errors.length);
      if (invalidRows.length) {
        const first = invalidRows[0];
        alert(`No se puede exportar: ${invalidRows.length} anotación(es) incompatibles. Primera: ${first.row.chunk_id}: ${first.errors.join('; ')}`);
        return;
      }
      const labeledRows = touchedRows.filter(isLabeled);
      const out = labeledRows.map(row => JSON.stringify(toAnnotationRecord(row))).join(String.fromCharCode(10));
      const blob = new Blob([out], { type: 'application/jsonl;charset=utf-8' });
      const a = document.createElement('a');
      a.href = URL.createObjectURL(blob);
      a.download = `${prefix}labeled_chunks.jsonl`;
      a.click();
      URL.revokeObjectURL(a.href);
    });

    window.addEventListener('resize', () => {
      if (currentRow()) meta.textContent = buildMetaLine(currentRow());
    });

    document.querySelectorAll('input[name=ann_type]').forEach(r => {
      r.addEventListener('change', () => { updateAnnotatorFieldVisibility(); saveCurrent(); });
    });
    document.getElementById('annotatorId').addEventListener('input', saveCurrent);
    document.getElementById('annotatorModel').addEventListener('input', saveCurrent);
    document.getElementById('skillFile').addEventListener('input', saveCurrent);
    scoreConfianza.addEventListener('input', saveCurrent);
    justificacion.addEventListener('input', saveCurrent);
    datasetFile.addEventListener('change', event => loadDatasetFile(event.target.files?.[0]));

    restoreAnnotatorMeta();
    updateAnnotatorFieldVisibility();
    renderLabels();
    bindGuideDialog();
    const embeddedRows = loadEmbeddedChunks();
    activateRows(embeddedRows, embeddedRows.length ? `${embeddedRows.length} chunks embebidos` : '');
  </script>
</body>
</html>"""

    return (
        html_template
        .replace('__DATA_SOURCE_BLOCK__', data_source_block)
        .replace('__DATA_MODE_HELP__', data_mode_help)
        .replace('__GUIDE_LABEL_CARDS__', guide_labels_html)
        .replace('__GUIDE_FLAG_ITEMS__', guide_flags_html)
    )


def recreate_frontend_with_dataset(dataset_path=None,
                                  html_file=FRONTEND_DIR / 'etiquetado_humano.html',
                                  guide_file=FRONTEND_DIR / 'guia_etiquetas.html',
                                  export_standalone_guide=False):
    """Recrea recursos del frontend para cualquier set de datos de chunks.

    - Reescribe etiquetado_humano.html completo con chunks + guia embebida en un solo archivo.
    - Opcionalmente genera guia_etiquetas.html externa si export_standalone_guide=True.
    """
    rows, source = load_chunks_dataset(dataset_path)

    html_file = Path(html_file)
    html_file.write_text(build_frontend_html(rows, embed_data=True), encoding='utf-8')

    if export_standalone_guide:
        write_labels_guide_html(guide_file)

    print(f'Dataset fuente     : {source}')
    print(f'Chunks incrustados : {len(rows)}')
    print(f'Frontend HTML      : {html_file}')
    if export_standalone_guide:
        print(f'Guia etiquetas     : {guide_file}')
    else:
        print('Guia embebida      : SI (mismo archivo HTML)')


def recreate_frontend_without_dataset(
    html_file=FRONTEND_DIR / 'etiquetado_humano_sin_datos.html',
):
    """Genera el frontend sin incluir chunks dentro del HTML.

    El archivo JSONL o JSON se selecciona localmente desde el navegador;
    sus datos no se envían a ningún servidor.
    """
    html_file = Path(html_file)
    html = build_frontend_html(rows=None, embed_data=False)
    if "id='embeddedChunks'" in html:
        raise ValueError('La variante sin datos no debe contener embeddedChunks.')
    html_file.write_text(html, encoding='utf-8')
    print(f'Frontend sin datos : {html_file}')
    print('Carga admitida     : JSONL o JSON local')
    return html_file


# Uso por defecto: genera las dos variantes del frontend.
recreate_frontend_with_dataset()
recreate_frontend_without_dataset()

In [9]:
# ── Paquete autocontenido para etiquetado con un LLM comercial ──────────────
# Esta celda elimina SOLO el contenido previo de para_equiquetado_LLM.
import hashlib
import json
import shutil
from pathlib import Path

PACKAGE_NAME = 'para_equiquetado_LLM'
ROOT_RESOLVED = ROOT.resolve()
PACKAGE_DIR = (ROOT_RESOLVED / PACKAGE_NAME).resolve()

# Guardia contra borrados fuera del destino exacto esperado.
if PACKAGE_DIR.parent != ROOT_RESOLVED or PACKAGE_DIR.name != PACKAGE_NAME:
    raise RuntimeError(f'Ruta de paquete insegura: {PACKAGE_DIR}')

PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
for item in PACKAGE_DIR.iterdir():
    if item.is_dir():
        shutil.rmtree(item)
    else:
        item.unlink()

SOURCE_FILES = {
    'clasificacion_moderacion_peru.md': ROOT_RESOLVED / 'modelos' / 'skills' / 'clasificacion_moderacion_peru.md',
    'taxonomia_moderacion.csv': ROOT_RESOLVED / 'datos' / 'processed' / 'taxonomia_moderacion.csv',
    'chunks_para_etiquetar.jsonl': ROOT_RESOLVED / 'datos' / 'processed' / 'chunks_para_etiquetar.jsonl',
}
missing = [str(path) for path in SOURCE_FILES.values() if not path.is_file()]
if missing:
    raise FileNotFoundError('Faltan archivos requeridos:\n' + '\n'.join(missing))

for output_name, source_path in SOURCE_FILES.items():
    shutil.copy2(source_path, PACKAGE_DIR / output_name)

LLM_PROMPT = r'''# Prompt operativo: etiquetado del corpus peruano

## Archivos de autoridad
Antes de clasificar, lee completos `clasificacion_moderacion_peru.md` y
`taxonomia_moderacion.csv`. Esos dos archivos son la autoridad normativa para esta tarea.
No sustituyas sus criterios por categorías aprendidas previamente, políticas genéricas de
toxicidad ni inferencias propias. Si algo no está sustentado por esos criterios, no inventes
una etiqueta: aplica `contexto_necesario` cuando corresponda.

El archivo de entrada es `chunks_para_etiquetar.jsonl`. Cada línea es un objeto independiente.
Usa el texto y el contexto del registro para aplicar, en orden, los siete pasos del skill.
La clasificación es multi-etiqueta y debe reflejar todas las categorías aplicables.

## Tarea
Etiqueta cada chunk sin omitir, duplicar ni reordenar registros. Conserva exactamente su
`chunk_id`. Evalúa cada caso con los criterios del skill; no reemplaces la evaluación por una
búsqueda simple de palabras clave ni por un clasificador heurístico. Nunca dejes `labels` vacío:
si no existe daño usa `seguro` o, únicamente cuando corresponda, `seguro_ironia_marcada`.
`seguro` no puede coexistir con una etiqueta de daño.

Usa un identificador constante de tres caracteres para todo el archivo: `CGT` para ChatGPT,
`GEM` para Gemini o `DSK` para DeepSeek. Registra en `annotator_model` el nombre exacto del
modelo utilizado y en `skill_file` el valor `clasificacion_moderacion_peru.md`.

## Salida obligatoria
Genera un archivo descargable `<id>_labeled_chunks.jsonl`, con un objeto JSON por línea y sin
bloques Markdown. Cada objeto debe contener exactamente estos campos:

- `chunk_id`
- `labels`
- `flags`
- `needs_review`
- `notes`
- `annotator_type` = `llm`
- `annotator_id`
- `annotator_model`
- `skill_file`
- `score_confianza`
- `justificacion`
- `annotated_at` en ISO 8601

No copies `text`, `video_id`, títulos, canal, tiempos, `text_hash` ni ningún otro campo del
chunk original. `ejemplo_formato_salida.jsonl` es únicamente una referencia estructural.

## Autoverificación antes de entregar
1. La cantidad de salidas coincide con la cantidad de chunks efectivamente procesados.
2. No hay `chunk_id` vacío, inventado o duplicado.
3. Todos los valores de `labels` y `flags` existen en la taxonomía.
4. `labels` nunca está vacío y `seguro` no coexiste con daño.
5. Cualquier flag, o confianza menor que 0.70, activa `needs_review=true`.
6. `ironia_ambigua` o `contexto_necesario` limitan `score_confianza` a 0.65.
7. La salida no contiene el texto ni otros campos fuente.
8. Cada justificación se basa en el criterio concreto del skill, no en categorías externas.

El corpus es grande. Si la plataforma no puede completarlo en una sola ejecución, trabaja en
partes consecutivas y entrega cada parte como JSONL, indicando con precisión el primer y último
`chunk_id` procesados. No afirmes que terminaste filas que no evaluaste.
'''
(PACKAGE_DIR / 'PROMPT_ETIQUETADO_LLM.md').write_text(
    LLM_PROMPT.strip() + '\n', encoding='utf-8'
)

EXAMPLE_RECORD = {
    'chunk_id': '<copiar_exactamente_el_chunk_id_de_entrada>',
    'labels': ['seguro'],
    'flags': [],
    'needs_review': False,
    'notes': '',
    'annotator_type': 'llm',
    'annotator_id': 'CGT',
    'annotator_model': '<nombre_exacto_del_modelo>',
    'skill_file': 'clasificacion_moderacion_peru.md',
    'score_confianza': 0.95,
    'justificacion': 'Texto informativo sin ataque; se aplica el criterio SEGURO del skill.',
    'annotated_at': '<fecha_hora_ISO_8601>',
}
(PACKAGE_DIR / 'ejemplo_formato_salida.jsonl').write_text(
    json.dumps(EXAMPLE_RECORD, ensure_ascii=False) + '\n', encoding='utf-8'
)

expected_names = set(SOURCE_FILES) | {
    'PROMPT_ETIQUETADO_LLM.md', 'ejemplo_formato_salida.jsonl'
}
actual_names = {path.name for path in PACKAGE_DIR.iterdir()}
if actual_names != expected_names:
    raise RuntimeError(f'Contenido inesperado en el paquete: {sorted(actual_names)}')

print('Paquete listo para carga manual:', PACKAGE_DIR)
for path in sorted(PACKAGE_DIR.iterdir()):
    digest = hashlib.sha256(path.read_bytes()).hexdigest()[:12]
    print(f'  {path.name:<40} {path.stat().st_size:>12,} bytes  sha256:{digest}')


Paquete listo para carga manual: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\para_equiquetado_LLM
  chunks_para_etiquetar.jsonl                63,133,033 bytes  sha256:eb90debf66d5
  clasificacion_moderacion_peru.md               33,620 bytes  sha256:45f9d3231a92
  ejemplo_formato_salida.jsonl                      435 bytes  sha256:c29eea090344
  PROMPT_ETIQUETADO_LLM.md                        2,989 bytes  sha256:47f85a096736
  taxonomia_moderacion.csv                        2,169 bytes  sha256:763c62f3d517
